# SFT — MedQA Calibration (Qwen2.5-3B-Instruct)

**Reproducible training and evaluation notebook** for the SFT part of [`skjdsk/medqa-calibration`](https://github.com/skjdsk/medqa-calibration).

- **Data:** 3,322 MedQA prompt/completion pairs retained from the team's candidate-selection pipeline; completions contain correct reasoning traces.
- **Training:** BF16 LoRA SFT, 1 epoch, rank 16 / alpha 32 / dropout 0.05, max length 2,048, effective batch size 4, completion-only loss.
- **Experiments:** Run A (seed 42, LR 5e-5); Run B (seed 42, LR 1e-4); independent final seed 123 (LR 1e-4). The final seed-42 adapter is **Run B**, not an additional retraining.
- **Evaluation:** Fixed 500-question MedQA validation for tuning; final BF16 direct-answer evaluation on MedQA test, MedMCQA validation, and PubMedQA labeled against an identically evaluated base model.
- **Published results:** [`results_summary/sft/`](https://github.com/skjdsk/medqa-calibration/tree/sft-results/results_summary/sft). The original 4-bit baseline under `results_summary/base/` is **not** directly comparable to the BF16 results here.

**Safety by default:** training, GPU evaluation, and results export are disabled unless deliberately enabled below. The previously trained adapters, full prediction CSVs, and checkpoints are on Google Drive. You do not need to rerun training to review the results. This notebook has no saved outputs or credentials.

## 1. Environment and data

Use a Google Colab **L4 GPU** with BF16 support for training/evaluation; a CPU runtime is sufficient for reading the saved summaries. Run the installation cell only when setting up a new runtime. If installation changes core libraries in an already-running session, restart the runtime before continuing.

In [ ]:
# Run once in a new Colab runtime, then restart if Colab requests it.
%pip install -q "trl==1.16.0" "transformers==4.57.6" "peft==0.18.0" "accelerate>=1.10.1,<2" "datasets>=4.7.0,<6" "bitsandbytes>=0.48,<0.50" "pandas>=2.0" "numpy>=1.24"

# Optional for a PEFT / torchao compatibility error:
# %pip uninstall -y torchao

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import json

from google.colab import drive

drive.mount("/content/drive")

REPO = Path("/content/medqa-sft-work")
if not (REPO / ".git").exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f"Not a Git repo, but directory is nonempty: {REPO}")
    subprocess.run([
        "git", "clone", "https://github.com/skjdsk/medqa-calibration.git", str(REPO)
    ], check=True)

os.chdir(REPO)
ROOT = Path("/content/drive/MyDrive/medqa-calibration/sft")
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATA_PATH = REPO / "outputs/medqa5000_L4_seed42_qpb2/selection_seed42/sft.jsonl"

assert DATA_PATH.exists(), f"Training data not found: {DATA_PATH}"
print("Repository:", REPO)
print("Training data:", DATA_PATH)
print("Drive root:", ROOT)

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer

with DATA_PATH.open(encoding="utf-8") as f:
    all_data = [json.loads(line) for line in f if line.strip()]

assert len(all_data) == 3322, f"Expected 3,322 examples, found {len(all_data)}"
assert all("prompt" in row and "completion" in row for row in all_data)

train_dataset = Dataset.from_list(all_data)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("SFT examples:", len(train_dataset))
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
print("BF16 available:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
print("Dataset columns:", train_dataset.column_names)

### Optional length diagnostics

The original experiment measured lengths using Qwen's chat template. The official training configuration uses `max_length=2048`; longer samples may be truncated. This diagnostic does not silently drop or modify examples.

In [ ]:
import numpy as np

sequence_lengths = np.asarray([
    len(tokenizer.apply_chat_template(row["prompt"] + row["completion"], tokenize=True))
    for row in all_data
])

print("Mean:", round(float(sequence_lengths.mean())))
print("Median:", int(np.median(sequence_lengths)))
print("P95:", int(np.percentile(sequence_lengths, 95)))
print("Maximum:", int(sequence_lengths.max()))
print("Over 2048 tokens:", int((sequence_lengths > 2048).sum()))

## 2. Experiment configuration

These are the actual runs documented in the original Colab session. Run B was selected after inspecting validation calibration metrics (exploratory choice, not a preregistered criterion); the final evaluation uses the existing Run B seed-42 adapter plus an independently trained seed-123 adapter.

Set a flag to `True` **only if you intentionally want to rerun** that expensive stage. Existing adapters are reused rather than overwritten.

In [ ]:
# Disabled by default to avoid accidental GPU usage or overwriting results.
RUN_TRAINING = False
RUN_VALIDATION = False
RUN_FINAL_EVALUATION = False
EXPORT_RESULTS_TO_REPO = False

RUNS = {
    "runA": {
        "out_dir": ROOT / "tuning_lr5e5", "lr": 5e-5,
        "seed": 42, "val_name": "sft_runA"
    },
    "runB": {
        "out_dir": ROOT / "tuning_lr1e4", "lr": 1e-4,
        "seed": 42, "val_name": "sft_runB"
    },
    "seed123": {
        "out_dir": ROOT / "final_seed123", "lr": 1e-4,
        "seed": 123, "val_name": "sft_seed123"
    },
}

for key, config in RUNS.items():
    print(key, "=>", config["out_dir"], "| LR", config["lr"], "| seed", config["seed"])

## 3. SFT training (GPU, opt-in)

Each run starts with a **fresh Qwen2.5-3B-Instruct base model**, not the LoRA adapter from the previous run. The runner uses the shared 3,322-example dataset, BF16 LoRA, the same batch/length configuration, and `get_last_checkpoint()` to resume interrupted training when possible. Checkpoints and final adapters are written to Google Drive.

In [ ]:
import gc
import time
from transformers import AutoModelForCausalLM
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig


def train_sft_run(run_key):
    cfg = RUNS[run_key]
    out_dir = cfg["out_dir"]
    adapter_dir = out_dir / "final_adapter"

    if (adapter_dir / "adapter_model.safetensors").exists() and \
       (adapter_dir / "adapter_config.json").exists():
        print(f"{run_key}: final adapter already exists; skip training: {adapter_dir}")
        return adapter_dir

    assert torch.cuda.is_available(), "Enable a GPU runtime to train."
    assert torch.cuda.is_bf16_supported(), "Training was performed on BF16-capable L4."
    out_dir.mkdir(parents=True, exist_ok=True)

    args = SFTConfig(
        output_dir=str(out_dir),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=cfg["lr"],
        max_length=2048,
        completion_only_loss=True,
        packing=False,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=20,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        save_only_model=False,
        report_to="none",
        seed=cfg["seed"],
        data_seed=cfg["seed"],
    )
    lora = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules="all-linear",
        bias="none",
        task_type="CAUSAL_LM",
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
    )
    model.config.use_cache = False
    trainer = None
    try:
        trainer = SFTTrainer(
            model=model,
            args=args,
            train_dataset=train_dataset,
            peft_config=lora,
            processing_class=tokenizer,
        )
        checkpoint = get_last_checkpoint(str(out_dir))
        print(f"{run_key}: start | checkpoint={checkpoint} | examples={len(train_dataset)}")
        start = time.time()
        result = trainer.train(resume_from_checkpoint=checkpoint)
        print(f"Completed {run_key}: step={trainer.state.global_step}, "
              f"loss={result.training_loss:.4f}, minutes={(time.time()-start)/60:.2f}")

        trainer.save_model(str(adapter_dir))
        tokenizer.save_pretrained(str(adapter_dir))
        assert (adapter_dir / "adapter_model.safetensors").exists()
        assert (adapter_dir / "adapter_config.json").exists()
        return adapter_dir
    finally:
        del trainer, model
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
if RUN_TRAINING:
    # Execute in historical order; each model trains independently.
    for run_key in ("runA", "runB", "seed123"):
        train_sft_run(run_key)
else:
    print("Training disabled. Existing adapters are not modified.")

### Original Run B recovery note

In the recorded Colab experiment, Run B completed training but its session crashed before the final-adapter save. The LoRA adapter and tokenizer files were recovered from `tuning_lr1e4/checkpoint-831` into `tuning_lr1e4/final_adapter`. **This is the source of the reported seed-42 final results.** The following CPU-only recovery is optional and will *not* overwrite an existing final adapter.

In [ ]:
import shutil


def restore_runB_adapter_if_needed():
    output = RUNS["runB"]["out_dir"] / "final_adapter"
    if (output / "adapter_model.safetensors").exists():
        print("Run B final adapter already exists; no recovery needed.")
        return output

    source = RUNS["runB"]["out_dir"] / "checkpoint-831"
    for required in ("adapter_model.safetensors", "adapter_config.json"):
        assert (source / required).exists(), f"Missing {required} in {source}"

    output.mkdir(parents=True, exist_ok=True)
    for filename in (
        "adapter_model.safetensors", "adapter_config.json", "tokenizer.json",
        "tokenizer_config.json", "special_tokens_map.json", "added_tokens.json",
        "chat_template.jinja", "vocab.json", "merges.txt"
    ):
        if (source / filename).exists():
            shutil.copy2(source / filename, output / filename)
    print("Recovered Run B adapter:", output)
    return output

# Call restore_runB_adapter_if_needed() only when recovery is necessary.

## 4. Validation: fixed 500 MedQA questions

Use `eval.evaluate` with `--medqa_subset val` and `--quantization none` for the base and each adapter. All validation prediction files remain on Drive. These artifacts were used to inspect LR sensitivity; do not treat the 500-question validation set as an independent final test set.

In [ ]:
def evaluate_validation(model_name, output_dir, adapter=None):
    output_dir.mkdir(parents=True, exist_ok=True)
    prediction = output_dir / f"preds_{model_name}_medqa4_val_direct.csv"
    if not prediction.exists():
        cmd = [
            sys.executable, "-m", "eval.evaluate",
            "--dataset", "medqa", "--medqa_subset", "val",
            "--model_name", model_name, "--quantization", "none",
            "--batch_size", "4", "--out_dir", str(output_dir)
        ]
        if adapter is not None:
            assert (adapter / "adapter_model.safetensors").exists(), adapter
            cmd += ["--adapter", str(adapter)]
        subprocess.run(cmd, cwd=REPO, check=True)
    else:
        print("Using existing validation predictions:", prediction)

    subprocess.run([
        sys.executable, "-m", "eval.metrics",
        "--pred_csv", str(prediction),
        "--out_dir", str(output_dir / "metrics"),
        "--n_bins", "15"
    ], cwd=REPO, check=True)
    return output_dir / "metrics/summary.csv"

In [ ]:
if RUN_VALIDATION:
    evaluate_validation("base_bf16_val", ROOT / "base_validation")
    for key in ("runA", "runB", "seed123"):
        cfg = RUNS[key]
        evaluate_validation(
            cfg["val_name"], cfg["out_dir"] / "validation",
            adapter=cfg["out_dir"] / "final_adapter"
        )
else:
    print("GPU validation disabled. Reading existing saved summaries below.")

In [ ]:
import pandas as pd
from IPython.display import display

validation_paths = {
    "base_bf16_val": ROOT / "base_validation/metrics/summary.csv",
    "sft_runA": RUNS["runA"]["out_dir"] / "validation/metrics/summary.csv",
    "sft_runB": RUNS["runB"]["out_dir"] / "validation/metrics/summary.csv",
    "sft_seed123": RUNS["seed123"]["out_dir"] / "validation/metrics/summary.csv",
}

available_validation = [pd.read_csv(p) for p in validation_paths.values() if p.exists()]
if available_validation:
    display(pd.concat(available_validation, ignore_index=True)[[
        "model_name", "accuracy", "ece", "brier", "nll",
        "hc_wrong_rate", "mean_confidence"
    ]].round(4))
else:
    print("No validation summary files at the documented Drive paths.")

## 5. Final evaluation: three datasets × three models

**Fixed comparison:**

| Run name | Weights | Precision |
|---|---|---|
| `base_bf16` | Original Qwen2.5-3B-Instruct | BF16, no quantization |
| `sft_seed42` | Run B (`tuning_lr1e4/final_adapter`) | BF16, no quantization |
| `sft_seed123` | Independent `final_seed123/final_adapter` | BF16, no quantization |

Each model is tested on MedQA **test** (1,273), MedMCQA **validation** (4,183), and PubMedQA **labeled** (1,000). The shared `eval.evaluate` defaults to direct-answer scoring. Predictions are saved per dataset, so interrupted evaluations can skip completed CSVs.

In [ ]:
FINAL_DIR = ROOT / "final_evaluation"
FINAL_MODELS = {
    "base_bf16": None,
    "sft_seed42": RUNS["runB"]["out_dir"] / "final_adapter",
    "sft_seed123": RUNS["seed123"]["out_dir"] / "final_adapter",
}
PREDICTION_DATASETS = {
    "medqa": "medqa4", "medmcqa": "medmcqa", "pubmedqa": "pubmedqa"
}


def evaluate_final_predictions():
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    for model_name, adapter in FINAL_MODELS.items():
        if adapter is not None:
            assert (adapter / "adapter_model.safetensors").exists(), adapter
        for dataset, stem in PREDICTION_DATASETS.items():
            prediction = FINAL_DIR / f"preds_{model_name}_{stem}_direct.csv"
            if prediction.exists():
                print("Skip existing:", prediction.name)
                continue
            cmd = [
                sys.executable, "-m", "eval.evaluate",
                "--dataset", dataset, "--model_name", model_name,
                "--quantization", "none", "--batch_size", "4",
                "--out_dir", str(FINAL_DIR)
            ]
            if adapter is not None:
                cmd += ["--adapter", str(adapter)]
            subprocess.run(cmd, cwd=REPO, check=True)

    prediction_files = sorted(FINAL_DIR.glob("preds_*.csv"))
    assert len(prediction_files) == 9, f"Expected 9 prediction files: got {len(prediction_files)}"
    subprocess.run([
        sys.executable, "-m", "eval.metrics",
        "--pred_csv", *map(str, prediction_files),
        "--out_dir", str(FINAL_DIR / "metrics"), "--n_bins", "15"
    ], cwd=REPO, check=True)
    return FINAL_DIR / "metrics/summary.csv"

In [ ]:
if RUN_FINAL_EVALUATION:
    evaluate_final_predictions()
else:
    print("GPU final evaluation disabled. Existing results are preserved.")

## 6. Read final results (CPU-only)

If Drive is available, read the complete evaluation results there. Otherwise fall back to the `results_summary/sft/summary.csv` already committed in the repository's `sft-results` branch. This does not rerun any GPU inference.

In [ ]:
FINAL_SUMMARY = FINAL_DIR / "metrics/summary.csv"
PUBLISHED_SUMMARY = REPO / "results_summary/sft/summary.csv"

if FINAL_SUMMARY.exists():
    selected_summary = FINAL_SUMMARY
elif PUBLISHED_SUMMARY.exists():
    selected_summary = PUBLISHED_SUMMARY
else:
    raise FileNotFoundError("Could not locate final summary in Drive or repo")

final_summary = pd.read_csv(selected_summary)
assert set(final_summary["model_name"]) == set(FINAL_MODELS)
assert set(final_summary["dataset"]) == set(PREDICTION_DATASETS)
assert len(final_summary) == 9

display(final_summary[[
    "model_name", "dataset", "accuracy", "ece", "brier", "nll", "hc_wrong_rate"
]].sort_values(["dataset", "model_name"]).round(4))

metrics = ["accuracy", "ece", "brier", "nll", "hc_wrong_rate"]
base = final_summary[final_summary.model_name == "base_bf16"].set_index("dataset")
sft_mean = (final_summary[final_summary.model_name.isin(["sft_seed42", "sft_seed123"])]
            .groupby("dataset")[metrics].mean())
comparison = pd.concat({
    "BF16_base": base[metrics],
    "SFT_2seed_mean": sft_mean,
    "SFT_minus_base": sft_mean - base[metrics]
}, axis=1)
display(comparison.round(4))

## 7. Export lightweight results for GitHub (opt-in)

The team's convention is to version small summaries, 15-bin tables, plots, and prediction metadata in `results_summary/sft/`. **Do not commit full per-question predictions, adapters, optimizer states, or checkpoints.** The repository `.gitignore` intentionally excludes `results/` and `outputs/`, but `results_summary/` is tracked.

In [ ]:
def export_lightweight_results():
    import shutil
    source = FINAL_DIR / "metrics"
    destination = REPO / "results_summary/sft"
    assert (source / "summary.csv").exists(), "Missing final metrics in Drive"
    destination.mkdir(parents=True, exist_ok=True)

    for filename in ("summary.csv", "summary.json"):
        shutil.copy2(source / filename, destination / filename)
    for pattern in ("bins_*.csv", "reliability_*.png"):
        for path in source.glob(pattern):
            shutil.copy2(path, destination / path.name)
    for path in FINAL_DIR.glob("preds_*_meta.json"):
        shutil.copy2(path, destination / path.name)

    print("Exported lightweight results to:", destination)
    print("Files:", len(list(destination.iterdir())))

if EXPORT_RESULTS_TO_REPO:
    export_lightweight_results()
else:
    print("Export disabled. Already-published results are not changed.")

## 8. Interpretation and reproducibility notes

- Accuracy changes are minimal on MedQA and MedMCQA; PubMedQA accuracy increases modestly. **NLL, Brier, and ECE decrease** for both SFT seeds on all three datasets.
- `hc_wrong_rate` means **wrong with confidence ≥ 0.90 as a fraction of all examples**, not the wrong fraction among only high-confidence examples.
- The base comparator is specifically `base_bf16`. Do **not** use `results_summary/base/` (older 4-bit baseline) as an equivalent reference.
- Letter probabilities are normalized across option letters. Inspect `letter_mass` in the metadata/prediction CSV when interpreting very high confidence.
- Two training seeds are insufficient to establish statistical significance. Run B's LR was selected after viewing validation results. The fixed 500-question validation set should not be repurposed as a completely independent test set.
- The original session contained a 64-example pilot, recovery/debug steps, and GitHub login/push commands. They have been intentionally excluded from the official *training/evaluation* workflow; the historical Run B checkpoint recovery is documented above.
- To submit the notebook, save it as **`notebooks/sft_train.ipynb`** on branch `sft-results` and commit only that file. The published 30 results files are already in the branch; merge via a pull request when the team is ready.